# Strong-lensing time-delay metric (TDC) versus the E(B-V) footprint threshold

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-28
- Kernel: conda_py313_opsim53
- Context: SCOC footprint-shrinking study + DESC metrics. This notebook extends the `10_DESCMAFDEPTHANDGALCOUNT` series to the **strong-lensing time-delay accuracy metric** `TdcMetric` (Liao et al. 2015, Time Delay Challenge), recomputed on each of the 7 dust-threshold footprint variants of the v5.3.6 simulations.
- This notebook: **11** of the series.
- Reference notebook (baseline only): `../02_MAF/science/DESC/02_TDC_TimeDelayAccuracy.ipynb` (Phil Marshall & Lynne Jones). Its last section (comparison of pre-computed summary values across the archived OpSim families) is not reproduced here.
- Companion notebooks of this series: `07_SNCounts_DESC_TaskForce_demo.ipynb` (per-pixel metric on the 7 runs with maps, histograms and consecutive differences, whose structure is reused here), `10_Sigma8tomo_demo.ipynb`.
- OpSim simulations analyzed (`/Users/dagoret/DATA/OpSim/`):
  - `shrink_fp_dust_0.050_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.080_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.120_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.150_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.199_v5.3.6_10yrs.db`
  - `baseline_v5.3.6_10yrs.db` (E(B-V) threshold 0.200)
  - `shrink_fp_dust_0.250_v5.3.6_10yrs.db`


## Notebook overview

**The metric.** `TdcMetric` is a 'complex' per-pixel metric: from the visits of a Healpix pixel it derives the night-to-night cadence, the season length and the campaign length, and plugs them in the Liao et al. (2015) scaling laws to get, for strongly lensed quasars, the time-delay accuracy A, precision P and success rate f (percent). It applies a limiting-magnitude cut to each visit and corrects the visit depth with the dust extinction of the pixel (`DustMap`). Post-processing then gives the sky area with high-accuracy delays (A < 0.04 %), the corresponding number of usable lenses and a combined distance-precision Figure of Merit (Coe & Moustakas 2009 approach).

**Important difference with the other metrics of the series: `TdcMetric` has no E(B-V) cut parameter.** The dust map only corrects the depth; it does not select a footprint. The E(B-V) threshold of a run acts here in two ways:
1. **through the visits**: the shrunk-footprint simulations schedule fewer visits (shorter seasons, sparser cadence) outside the smaller WFD area, which degrades A, P, f in those pixels. This is what MAF measures, and it needs no parameter.
2. **through the definition of the footprint** on which the summary numbers are computed. This is a post-processing choice, made here with a mask `E(B-V) < threshold` on the Healpix map (`EBV_MASK_MODE`, Section 3): `'run'` uses the threshold of each run (default), `'fixed'` uses `FIXED_LIM_EBV = 0.2` for all runs, `'none'` uses every valid pixel of the sky (the choice of the reference notebook). Both the footprint-masked and the all-valid-pixel statistics are tabulated in Section 7.

**Workflow.** MAF is run **once per simulation** on the full sky (`HealpixSlicer`, `nside = 64`) and the six TDC quantities (accuracy, precision, rate, cadence, season, campaign) are saved as Healpix maps in a `.npz` cache. All the analysis below (masks, tables, figures) is done from these maps, so changing `EBV_MASK_MODE` or the accuracy threshold does not require rerunning MAF.

**Figures** (presentation model: `../07_variateEVmV/01_FOMNv_HealpixDiff_ShrinkFPDust.ipynb`):
1. Healpix maps of the accuracy and of the success rate for the 7 runs on a common color scale, and the map of the high-accuracy region.
2. Area-weighted histograms of accuracy, precision and rate.
3. Maps of the differences between consecutive E(B-V) thresholds, with the pixels gained or lost by the footprint tabulated separately.
4. High-accuracy area, number of lenses, distance precision, mean rate and accuracy versus the E(B-V) threshold, raw and normalized to the baseline.
5. A consistency check against the reference notebook.

**Differences of setup with respect to the reference notebook.** The reference notebook uses all visits (`constraint=None`) of a 10-year baseline. Here, to follow the convention of the series (README of this folder), the first 10 years and non-DDF visits are selected: `night <= 3653.5 and scheduler_note not like 'DD%'` (the baseline file has 11 years). The constraint is editable in Section 3.

## Practical note before running

`TdcMetric` at `nside = 64` (49152 pixels) is slow, several minutes per simulation. The maps of each run are cached in `data_11_TDC/`; later executions only read them. The E(B-V) map of the pixels is obtained with MAF's `DustMap` and cached as well.

## 1. Imports

In [ ]:
import os
import time
from os.path import join, isfile

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf

print("rubin_sim version:", rubin_sim.__version__)

## 2. Output directories

In [ ]:
NB_TAG = "TDC"
data_dir = f"data_11_{NB_TAG}"
figs_dir = f"figs_11_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

resultsDb = maf.db.ResultsDb(out_dir=data_dir)

## 3. Configuration

In [ ]:
# Directory holding the OpSim databases
OPSIM_DIR = "/Users/dagoret/DATA/OpSim"

# (run_name, E(B-V) threshold that defines the footprint of the run), sorted by increasing threshold
# - identical list to the other notebooks of this series
RUNS_INFO = [
    ("shrink_fp_dust_0.050_v5.3.6_10yrs", 0.050),
    ("shrink_fp_dust_0.080_v5.3.6_10yrs", 0.080),
    ("shrink_fp_dust_0.120_v5.3.6_10yrs", 0.120),
    ("shrink_fp_dust_0.150_v5.3.6_10yrs", 0.150),
    ("shrink_fp_dust_0.199_v5.3.6_10yrs", 0.199),
    ("baseline_v5.3.6_10yrs", 0.200),
    ("shrink_fp_dust_0.250_v5.3.6_10yrs", 0.250),
]
RUN_NAMES = [r for r, _ in RUNS_INFO]
DUST_THRESH = dict(RUNS_INFO)
REF_RUN = "baseline_v5.3.6_10yrs"
i_ref = RUN_NAMES.index(REF_RUN)


def get_db_path(run_name):
    # Look in OPSIM_DIR, then in OPSIM_DIR/sim_baseline (where the baseline may be stored)
    fname = run_name + ".db"
    candidates = [join(OPSIM_DIR, fname), join(OPSIM_DIR, "sim_baseline", fname)]
    for c in candidates:
        if isfile(c):
            return c
    raise FileNotFoundError(f"OpSim db not found for run {run_name}. Tried: {candidates}")


for run_name, dust in RUNS_INFO:
    try:
        path = get_db_path(run_name)
    except FileNotFoundError:
        path = "NOT FOUND (only a problem if there is no cached result, see Section 5)"
    print(f"{run_name:40s} E(B-V) < {dust:.3f}  ->  {path}")

In [ ]:
NSIDE = 64  # same nside as the reference notebook
npix = hp.nside2npix(NSIDE)
pix_area = hp.nside2pixarea(NSIDE, degrees=True)

# The six quantities of TdcMetric, and the keys of the dict it returns in each pixel
TDC_KEYS = ["accuracy", "precision", "rate", "cadence", "season", "campaign"]
TDC_DICT_KEYS = {
    "accuracy": "accuracy",
    "precision": "precision",
    "rate": "rate",
    "cadence": "cadence (days)",
    "season": "season (months)",
    "campaign": "campaign",
}
UNITS = {
    "accuracy": "%",
    "precision": "%",
    "rate": "%",
    "cadence": "days",
    "season": "months",
    "campaign": "years",
}

# Visits given to MAF: first 10 years, non-DDF (convention of the series; the reference notebook uses all visits).
# If you change this constraint, change CONSTRAINT_TAG too (it is part of the cache file names).
SQL_CONSTRAINT = f'night <= {10 * 365.25 + 0.5} and scheduler_note not like "DD%"'
CONSTRAINT_TAG = "nonDD10yr"

# High-accuracy selection and distance-precision FoM (constants of the reference notebook)
ACC_THRESHOLD = 0.04  # % ; 5 times better than the 0.2 % threshold of Hojjati & Linder (2014)
MODELING_ERROR_PCT = 4.0  # lens-modeling systematic error, added in quadrature to the precision
REF_AREA_SQDEG = 18000.0  # TDC1 reference survey area
REF_RATE_PCT = 30.0  # TDC1 reference success rate
REF_N_LENSES = 400  # TDC1 reference lens yield

# Footprint on which the summary numbers are computed (post-processing mask on E(B-V), see the overview):
#   'run'   -> pixels with E(B-V) < threshold of the run
#   'fixed' -> pixels with E(B-V) < FIXED_LIM_EBV, the same for all runs
#   'none'  -> all the valid pixels of the sky
EBV_MASK_MODE = "run"
FIXED_LIM_EBV = 0.2

FORCE_RECOMPUTE = False  # True -> ignore the cached maps and rerun MAF

print(f"nside={NSIDE} -> npix={npix}, pixel area = {pix_area:.4f} deg^2")
print("SQL constraint:", SQL_CONSTRAINT)


def footprint_ebv_cut(run_name):
    # E(B-V) upper limit of the footprint mask for this run
    if EBV_MASK_MODE == "run":
        return DUST_THRESH[run_name]
    if EBV_MASK_MODE == "fixed":
        return FIXED_LIM_EBV
    if EBV_MASK_MODE == "none":
        return np.inf
    raise ValueError(f"EBV_MASK_MODE must be run, fixed or none, got {EBV_MASK_MODE!r}")

## 4. Helper functions

In [ ]:
def run_label(run_name):
    label = f"E(B-V) < {DUST_THRESH[run_name]:.3f}"
    return label + " (baseline)" if run_name.startswith("baseline") else label


def save_fig(fig, name):
    base = join(figs_dir, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")


def maps_cache_path(run_name):
    tag = run_name.replace(".", "_")
    return join(data_dir, f"TDC_maps_{tag}_nside{NSIDE}_{CONSTRAINT_TAG}.npz")


def tdc_maps_from_bundle(bundle):
    # TdcMetric stores in each valid pixel a dict (accuracy, precision, rate, cadence (days),
    # season (months), campaign). Unpack it into six full-sky Healpix maps (NaN where masked).
    x = bundle.metric_values
    if len(x) != npix:
        raise ValueError(f"metric_values has {len(x)} slices, expected {npix} (full-sky Healpix map)")
    mask = np.ma.getmaskarray(x)
    maps = {k: np.full(npix, np.nan) for k in TDC_KEYS}
    n_bad = 0
    for ipix in np.where(~mask)[0]:
        v = x.data[ipix]
        if not isinstance(v, dict):
            n_bad += 1
            continue
        for k in TDC_KEYS:
            maps[k][ipix] = v.get(TDC_DICT_KEYS[k], np.nan)
    print(f"  valid pixels: {int((~mask).sum())} / {npix}   (unexpected non-dict values: {n_bad})")
    return maps


dustmap = maf.DustMap(nside=NSIDE, interp=False)


def compute_run_maps(run_name):
    dbpath = get_db_path(run_name)
    print(f"[{run_name}] running TdcMetric on {dbpath}")
    slicer = maf.HealpixSlicer(nside=NSIDE, use_cache=False)
    bundle = maf.MetricBundle(
        maf.TdcMetric(metric_name="TDC"),
        slicer,
        constraint=SQL_CONSTRAINT,
        run_name=run_name,
        maps_list=[dustmap],
    )
    group = maf.MetricBundleGroup({"TDC": bundle}, dbpath, out_dir=data_dir, results_db=resultsDb)
    group.run_all()
    return tdc_maps_from_bundle(bundle)


def load_or_compute_maps(run_name):
    path = maps_cache_path(run_name)
    if isfile(path) and not FORCE_RECOMPUTE:
        print(f"[{run_name}] loaded cached maps: {path}")
        with np.load(path) as z:
            return {k: z[k] for k in TDC_KEYS}
    maps = compute_run_maps(run_name)
    np.savez_compressed(path, **maps)
    print(f"[{run_name}] saved: {path}")
    return maps


def get_ebv_map():
    # E(B-V) of every Healpix pixel, from the MAF DustMap (same map the metric uses for its m5 correction)
    path = join(data_dir, f"ebv_map_nside{NSIDE}.npy")
    if isfile(path):
        return np.load(path)
    slicer = maf.HealpixSlicer(nside=NSIDE, use_cache=False)
    try:
        slice_points = dustmap.run(slicer.slice_points)
        ebv = np.asarray(slice_points["ebv"], dtype=float)
    except Exception as err:
        raise RuntimeError(
            "Could not get E(B-V) per pixel from maf.DustMap.run(slicer.slice_points). "
            "Set EBV_MASK_MODE = none to skip the footprint mask, or adapt this function."
        ) from err
    if ebv.size != npix:
        raise RuntimeError(f"E(B-V) map has {ebv.size} pixels, expected {npix}")
    np.save(path, ebv)
    return ebv


def summarize(maps, mask):
    # Sky-averaged values and high-accuracy-region summary for the pixels selected by mask.
    # Same definitions as summarize_high_accuracy_region / compute_distance_precision_fom of the reference notebook.
    n_pix = int(mask.sum())
    out = {"n_pix": n_pix, "area_deg2": n_pix * pix_area}
    for k in TDC_KEYS:
        out["mean_" + k] = float(np.nanmean(maps[k][mask])) if n_pix else np.nan
    out["median_accuracy"] = float(np.nanmedian(maps["accuracy"][mask])) if n_pix else np.nan

    hi = mask & (maps["accuracy"] < ACC_THRESHOLD)
    n_hi = int(hi.sum())
    out["hi_n_pix"] = n_hi
    out["hi_area_deg2"] = n_hi * pix_area
    out["hi_fraction_pct"] = 100.0 * n_hi / n_pix if n_pix else np.nan
    for k in ("cadence", "season", "campaign"):
        out["hi_median_" + k] = float(np.median(maps[k][hi])) if n_hi else np.nan

    if n_hi:
        prec_lens = float(np.hypot(np.mean(maps["precision"][hi]), MODELING_ERROR_PCT))
        n_lenses = int(
            (out["hi_area_deg2"] / REF_AREA_SQDEG) * (np.mean(maps["rate"][hi]) / REF_RATE_PCT) * REF_N_LENSES
        )
        dist_prec = prec_lens / np.sqrt(n_lenses) if n_lenses > 0 else np.nan
    else:
        prec_lens, n_lenses, dist_prec = np.nan, 0, np.nan
    out["precision_per_lens_pct"] = prec_lens
    out["n_lenses"] = n_lenses
    out["distance_precision_pct"] = dist_prec
    return out


def masked_map(vals, mask):
    # Healpix map with hp.UNSEEN outside the mask (drawn in the bad-pixel color by healpy)
    return np.where(mask & np.isfinite(vals), vals, hp.UNSEEN)

## 5. Compute (or reload) the TDC maps of the 7 runs

In [ ]:
t0 = time.time()
maps_by_run = {run_name: load_or_compute_maps(run_name) for run_name, _ in RUNS_INFO}
print(f"Done in {time.time() - t0:.1f} s")

## 6. E(B-V) map and footprint masks

In [ ]:
ebv = get_ebv_map() if EBV_MASK_MODE != "none" else np.zeros(npix)
print(
    f"E(B-V) map: min = {np.nanmin(ebv):.3f}, max = {np.nanmax(ebv):.3f}, NaN pixels = {int(np.isnan(ebv).sum())}"
)

valid_mask = {r: np.isfinite(maps_by_run[r]["accuracy"]) for r in RUN_NAMES}
fp_mask = {r: valid_mask[r] & (ebv < footprint_ebv_cut(r)) for r in RUN_NAMES}

print(f"Footprint mask mode: {EBV_MASK_MODE}")
for r in RUN_NAMES:
    print(
        f"{r:40s} valid: {valid_mask[r].sum():6d} px ({valid_mask[r].sum() * pix_area:8.0f} deg^2)   "
        f"footprint: {fp_mask[r].sum():6d} px ({fp_mask[r].sum() * pix_area:8.0f} deg^2)"
    )

## 7. Summary table

Two versions: `stats_fp` (pixels of the footprint mask, the main one, used in the figures) and `stats_all` (all valid pixels of the sky, the choice of the reference notebook). The high-accuracy region is nearly identical in both, since pixels with poor sampling do not pass the accuracy cut; the sky averages differ.

In [ ]:
stats_fp = pd.DataFrame([summarize(maps_by_run[r], fp_mask[r]) for r in RUN_NAMES], index=RUN_NAMES)
stats_all = pd.DataFrame([summarize(maps_by_run[r], valid_mask[r]) for r in RUN_NAMES], index=RUN_NAMES)
for df in (stats_fp, stats_all):
    df.insert(0, "ebv_threshold", [DUST_THRESH[r] for r in RUN_NAMES])

stats_fp.to_csv(join(data_dir, "TDC_stats_footprint_mask.csv"))
stats_all.to_csv(join(data_dir, "TDC_stats_all_valid_pixels.csv"))
print("Saved: TDC_stats_footprint_mask.csv and TDC_stats_all_valid_pixels.csv in", data_dir)
print()
print(f"--- stats_fp: footprint mask (mode {EBV_MASK_MODE}) ---")
print(stats_fp.round(3).T.to_string())

In [ ]:
print("--- stats_all: all valid pixels of the sky ---")
print(stats_all.round(3).T.to_string())

## 8. Healpix maps for the 7 runs

Common color scale for all runs (1st to 99th percentile of the pooled footprint pixels); gray = pixels outside the footprint or where the metric is masked.

In [ ]:
def pooled_range(q, lo=1.0, hi=99.0):
    pooled = np.concatenate([maps_by_run[r][q][fp_mask[r]] for r in RUN_NAMES])
    return tuple(float(v) for v in np.nanpercentile(pooled, [lo, hi]))


def plot_run_maps(q_maps, title, unit, vmin, vmax, cmap, name, ncols=2):
    nrows = int(np.ceil(len(RUN_NAMES) / ncols))
    fig = plt.figure(figsize=(7 * ncols, 3.6 * nrows))
    for i, run_name in enumerate(RUN_NAMES):
        hp.mollview(
            q_maps[run_name],
            fig=fig.number,
            sub=(nrows, ncols, i + 1),
            title=run_label(run_name),
            unit=unit,
            min=vmin,
            max=vmax,
            cmap=cmap,
        )
    fig.suptitle(title, y=1.0, fontsize=14)
    save_fig(fig, name)
    plt.show()


MAP_QUANTITIES = ["accuracy", "rate"]
for q in MAP_QUANTITIES:
    vmin, vmax = pooled_range(q)
    q_maps = {r: masked_map(maps_by_run[r][q], fp_mask[r]) for r in RUN_NAMES}
    plot_run_maps(q_maps, f"TDC {q} ({UNITS[q]})", UNITS[q], vmin, vmax, "viridis", f"TDC_{q}_maps_allruns")

### 8.1 High-accuracy region (accuracy below the threshold)

Green = pixels of the footprint with accuracy A < `ACC_THRESHOLD`, from which the number of usable lenses is estimated; yellow = other footprint pixels.

In [ ]:
hi_maps = {
    r: masked_map((maps_by_run[r]["accuracy"] < ACC_THRESHOLD).astype(float), fp_mask[r]) for r in RUN_NAMES
}
plot_run_maps(
    hi_maps,
    f"TDC high-accuracy region (A < {ACC_THRESHOLD} %)",
    "high accuracy",
    0.0,
    1.0,
    "summer_r",
    "TDC_highaccuracy_maps_allruns",
)

## 9. Histograms

Area-weighted (each pixel counts for its solid angle in deg^2) histograms of the footprint pixels, one curve per run.

In [ ]:
colors = plt.cm.viridis(np.linspace(0.0, 0.95, len(RUNS_INFO)))
HIST_QUANTITIES = ["accuracy", "precision", "rate"]

fig, axs = plt.subplots(1, len(HIST_QUANTITIES), figsize=(6 * len(HIST_QUANTITIES), 5))
for ax, q in zip(axs, HIST_QUANTITIES):
    lo, hi = np.nanpercentile(np.concatenate([maps_by_run[r][q][fp_mask[r]] for r in RUN_NAMES]), [0.5, 99.5])
    bins = np.linspace(lo, hi, 61)
    for run_name, c in zip(RUN_NAMES, colors):
        vals = maps_by_run[run_name][q][fp_mask[run_name]]
        ax.hist(
            vals,
            bins=bins,
            weights=np.full(vals.size, pix_area),
            histtype="step",
            color=c,
            label=run_label(run_name),
        )
    ax.set_yscale("log")
    ax.set_xlabel(f"TDC {q} ({UNITS[q]})")
    ax.set_ylabel("Area per bin (deg^2)")
    ax.grid(alpha=0.3)
axs[0].legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "TDC_histograms_allruns")
plt.show()

## 10. Differences between consecutive E(B-V) thresholds

Differences `(threshold i+1) - (threshold i)` of the maps of the metric, on the pixels valid in **both** footprints (`0.199 -> baseline (0.200)` shows the run-to-run scheduler fluctuations at almost the same footprint). The pixels gained or lost by the footprint are tabulated separately.

In [ ]:
pairs = list(zip(RUN_NAMES[:-1], RUN_NAMES[1:]))

rows = []
for lo_run, hi_run in pairs:
    gained = fp_mask[hi_run] & ~fp_mask[lo_run]
    lost = fp_mask[lo_run] & ~fp_mask[hi_run]
    both = fp_mask[hi_run] & fp_mask[lo_run]
    rows.append(
        {
            "difference": f"{DUST_THRESH[hi_run]:.3f} - {DUST_THRESH[lo_run]:.3f}",
            "gained_px": int(gained.sum()),
            "lost_px": int(lost.sum()),
            "common_px": int(both.sum()),
            "gained_deg2": gained.sum() * pix_area,
            "lost_deg2": lost.sum() * pix_area,
        }
    )
print(pd.DataFrame(rows).round(1).to_string(index=False))


def plot_diff_maps(q, name, ncols=2):
    diffs = []
    for lo_run, hi_run in pairs:
        both = fp_mask[lo_run] & fp_mask[hi_run]
        d = np.full(npix, hp.UNSEEN)
        d[both] = maps_by_run[hi_run][q][both] - maps_by_run[lo_run][q][both]
        diffs.append(d)
    pooled = np.concatenate([d[d != hp.UNSEEN] for d in diffs])
    lim = float(np.nanpercentile(np.abs(pooled), 98)) if pooled.size else 1.0
    lim = lim if lim > 0 else 1.0
    nrows = int(np.ceil(len(pairs) / ncols))
    fig = plt.figure(figsize=(7 * ncols, 3.6 * nrows))
    for i, ((lo_run, hi_run), d) in enumerate(zip(pairs, diffs)):
        hp.mollview(
            d,
            fig=fig.number,
            sub=(nrows, ncols, i + 1),
            title=f"{DUST_THRESH[hi_run]:.3f} - {DUST_THRESH[lo_run]:.3f}",
            unit=UNITS[q],
            min=-lim,
            max=lim,
            cmap="RdBu_r",
        )
    fig.suptitle(
        f"TDC {q}: difference between consecutive E(B-V) thresholds ({UNITS[q]})", y=1.0, fontsize=14
    )
    save_fig(fig, name)
    plt.show()


DIFF_QUANTITIES = ["accuracy", "rate"]
for q in DIFF_QUANTITIES:
    plot_diff_maps(q, f"TDC_{q}_diffmaps_consecutive")

## 11. Summary quantities versus the E(B-V) footprint threshold

One point per run (footprint-masked statistics of Section 7). The vertical dashed line marks the baseline threshold. The distance precision is the proxy Figure of Merit of the reference notebook; Treu & Marshall (2016) quote an LSST-era target of 0.4-0.7 %.

In [ ]:
x = np.array([DUST_THRESH[r] for r in RUN_NAMES])

SUMMARY_PANELS = [
    ("hi_area_deg2", "High-accuracy area (deg^2)"),
    ("n_lenses", "Number of lenses (high-accuracy sample)"),
    ("distance_precision_pct", "Distance precision (%)"),
    ("hi_fraction_pct", "High-accuracy fraction of the footprint (%)"),
    ("mean_rate", "Mean success rate f (%)"),
    ("mean_accuracy", "Mean accuracy A (%)"),
]


def plot_summary_vs_threshold(stats, normalized, name):
    fig, axs = plt.subplots(2, 3, figsize=(15, 8))
    for ax, (col, ylabel) in zip(axs.ravel(), SUMMARY_PANELS):
        y = stats[col].to_numpy(dtype=float)
        if normalized:
            y = y / y[i_ref]
            ax.axhline(1.0, color="gray", ls="--", linewidth=0.8)
            ylabel = ylabel + " / baseline"
        ax.plot(x, y, marker="o", color="C0")
        ax.axvline(x[i_ref], color="gray", ls="--", linewidth=0.8)
        ax.set_xlabel("E(B-V) threshold")
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)
    suffix = " (normalized to the baseline)" if normalized else ""
    fig.suptitle("TDC metric summaries versus the E(B-V) footprint threshold" + suffix, y=1.0)
    fig.tight_layout()
    save_fig(fig, name)
    plt.show()


plot_summary_vs_threshold(stats_fp, False, "TDC_summary_vs_dust_threshold")

### 11.1 Same plot, normalized to the baseline run

In [ ]:
plot_summary_vs_threshold(stats_fp, True, "TDC_summary_vs_dust_threshold_norm")

### 11.2 Cadence, season and campaign length of the high-accuracy region

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, k in zip(axs, ("cadence", "season", "campaign")):
    ax.plot(x, stats_fp["hi_median_" + k].to_numpy(dtype=float), marker="o", color="C1")
    ax.axvline(x[i_ref], color="gray", ls="--", linewidth=0.8)
    ax.set_xlabel("E(B-V) threshold")
    ax.set_ylabel(f"Median {k} in the high-accuracy region ({UNITS[k]})")
    ax.grid(alpha=0.3)
fig.tight_layout()
save_fig(fig, "TDC_highaccuracy_cadence_season_campaign_vs_dust_threshold")
plt.show()

## 12. Consistency check against the reference notebook

The baseline numbers of the reference notebook (fraction and area of the high-accuracy region, median cadence/season/campaign there, number of lenses, distance precision) are printed below for the **all-valid-pixel** statistics, the choice of the reference notebook. An exact match also requires `constraint=None` and the 10-year baseline file in the reference notebook, whereas this notebook uses the first 10 years, non-DDF visits of `baseline_v5.3.6_11yrs` (Section 3): small differences are expected.

In [ ]:
b = stats_all.loc[REF_RUN]
print("Baseline run:", REF_RUN)
print(
    "Fraction of total survey area providing high accuracy time delays =", round(b["hi_fraction_pct"], 1), "%"
)
print("Median night-to-night cadence in high accuracy regions =", round(b["hi_median_cadence"], 1), "days")
print("Median season length in high accuracy regions =", round(b["hi_median_season"], 1), "months")
print("Median campaign length in high accuracy regions =", int(b["hi_median_campaign"]), "years")
print("Area of sky providing high accuracy time delays =", int(b["hi_area_deg2"]), "sq deg")
print(
    "Mean precision per lens in high accuracy sample, including modeling error =",
    round(b["precision_per_lens_pct"], 2),
    "%",
)
print("Number of lenses in high accuracy sample =", int(b["n_lenses"]))
print("Maximum combined percentage distance precision =", round(b["distance_precision_pct"], 2), "%")

## Caveats

- **The E(B-V) footprint mask is a post-processing choice.** `TdcMetric` has no dust-cut parameter, so unlike notebooks 01, 03, 06, 07 and 10 the threshold is not given to the metric. The footprint of a `shrink_fp_dust` run is not exactly `E(B-V) < threshold` (the scheduler footprint has additional smoothing and other regions), and the `DustMap` used here is not necessarily the map the scheduler used to build it, so pixels along the boundary may differ. The high-accuracy region, which requires long seasons and dense sampling, is much less sensitive to this choice than the sky averages (compare `stats_fp` and `stats_all`).
- **Sky averages mix well-sampled and poorly-sampled pixels.** Outside the WFD, `TdcMetric` may still return values from a few visits; the means of A, P, f over the footprint are then dominated by how many such pixels the mask lets in. The high-accuracy region and the lens counts are the more robust summaries.
- **Cadence model.** The TDC1 heuristics were derived for a nearly constant single-filter survey strategy; rolling cadence distributes visits unevenly over time, which `TdcMetric` (median season length, mean cadence) may not capture well. The differences between runs should be read as relative, not absolute, changes. All filters are treated as if they were a single-filter monitoring.
- **Scheduler fluctuations.** `0.199` and `baseline (0.200)` have almost the same nominal footprint: the difference between their results gives the scale of the run-to-run fluctuations at fixed threshold, to compare with the change between the more widely separated thresholds.
- **Selection of visits.** Non-DDF, first 10 years (see Section 3); the reference notebook uses all visits. `TdcMetric` also applies its own limiting-magnitude cuts per visit.
- **Not reproduced.** The last section of the reference notebook (cross-run comparison of the archived summary values of the 'SL TDC' metric set over many OpSim families) does not depend on the E(B-V) threshold and is left in the reference notebook.

## References
- Liao, K. et al. 2015, 'Strong lens time delay challenge. II. Results of TDC1', arXiv:1409.1254 - the accuracy/precision/success-rate scaling laws used by `TdcMetric`.
- Coe, D. & Moustakas, L. 2009, ApJ 706, 45 - combination of many lenses into a distance precision.
- Hojjati, A. & Linder, E. 2014, PRD 90, 123501 - accuracy requirement on time delays.
- Treu, T. & Marshall, P. 2016, A&ARv 24, 11, arXiv:1605.05333 - LSST-era target for the time-delay distance precision.
- Lochner, M. et al. 2018, 'Optimizing LSST Observing Strategy for Dark Energy Science', arXiv:1808.00006
- Bianco, F. B. et al. 2022, ApJS 258, 1 (SCOC context)
- `rubin_sim` source: https://github.com/lsst/rubin_sim (`rubin_sim/maf/metrics/season_metrics.py`, `TdcMetric`)
- Reference notebook `../02_MAF/science/DESC/02_TDC_TimeDelayAccuracy.ipynb`; structure of this series: `07_SNCounts_DESC_TaskForce_demo.ipynb`, `../07_variateEVmV/01_FOMNv_HealpixDiff_ShrinkFPDust.ipynb`.
- `shrink_fp_dust_*.db` simulations: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/shrink_fp/